# California Housing Investment Analysis

## Data Science Programming Languages and Tools

**Team Leader:** [INSERT NAME]  
**Group Members:**  
- [INSERT NAME]
- [INSERT NAME]
- [INSERT NAME]

### Objective
This project analyzes the California Housing dataset to identify areas that may be interesting for different types of real-estate investors. The analysis covers data understanding, exploratory analysis, client-specific investment criteria, feature engineering, predictive modeling, model interpretation, value screening, error analysis, and critical discussion.

> **Important:** Model results and candidate areas are generated by the notebook from the dataset and should not be treated as direct investment advice.


# Part 1 — Data loading and understanding

Each observation in the California Housing dataset represents a **census block group**, not an individual property. The target `MedHouseVal` represents the median house value for that block group, expressed in units of **$100,000**.

Variables that can be useful for investment-related screening include median income, average rooms, average occupancy, house age, population, latitude, and longitude.

Important real-estate variables missing from the dataset include property size, bathrooms, rental prices, historical price appreciation, mortgage rates, property taxes, crime, school quality, vacancy, and individual property condition.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
display(df.head())


### Answers

1. **Unit of observation:** a California census block group.
2. **Individual property?** No. Each row is an aggregated geographic area.
3. **Target:** median house value of the block group.
4. **Target unit:** hundreds of thousands of US dollars.
5. **Useful variables:** `MedInc`, `AveRooms`, `AveOccup`, `HouseAge`, `Population`, `Latitude`, and `Longitude`.
6. **Missing variables:** property size, bathrooms, rent, historical prices, taxes, financing costs, crime, schools, vacancy, and property condition.


# Part 2 — Initial data inspection

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isnull().sum().to_frame("missing"))

print("\nDescriptive statistics:")
display(df.describe().T)


In [ ]:
# Identify minimum and maximum values
extremes = pd.DataFrame({
    "min": df.min(),
    "max": df.max()
})
display(extremes)

# Histograms
df.hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()


### Interpretation

The dataset contains 20,640 observations and no missing values. Several variables show skewed distributions and some observations are extreme, especially variables such as average rooms, average bedrooms, population, and average occupancy.

The target has an important limitation: median house value is capped at an upper level in this dataset. This can make very expensive areas harder to distinguish and can affect model errors for high-value observations.


# Part 3 — Exploratory Data Analysis

In [ ]:
# 1. Median income vs house value
plt.figure(figsize=(8, 6))
plt.scatter(df["MedInc"], df["MedHouseVal"], alpha=0.2)
plt.xlabel("Median Income")
plt.ylabel("Median House Value")
plt.title("Median Income vs Median House Value")
plt.show()

# 2. Average rooms vs house value
plt.figure(figsize=(8, 6))
plt.scatter(df["AveRooms"], df["MedHouseVal"], alpha=0.2)
plt.xlabel("Average Rooms")
plt.ylabel("Median House Value")
plt.title("Average Rooms vs Median House Value")
plt.show()

# 3. Average occupancy vs house value
plt.figure(figsize=(8, 6))
plt.scatter(df["AveOccup"], df["MedHouseVal"], alpha=0.2)
plt.xlabel("Average Occupancy")
plt.ylabel("Median House Value")
plt.title("Average Occupancy vs Median House Value")
plt.show()


In [ ]:
# Correlation analysis
correlation = df.corr(numeric_only=True)
display(correlation["MedHouseVal"].sort_values(ascending=False))

plt.figure(figsize=(10, 8))
plt.imshow(correlation, cmap="coolwarm", aspect="auto")
plt.colorbar()
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=90)
plt.yticks(range(len(correlation.columns)), correlation.columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# Geographic visualization
plt.figure(figsize=(9, 7))
plt.scatter(
    df["Longitude"],
    df["Latitude"],
    c=df["MedHouseVal"],
    alpha=0.4
)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("California Housing Values by Geographic Location")
plt.colorbar(label="Median House Value")
plt.show()


### Interpretation

`MedInc` is expected to have the strongest positive association with median house value. Not all relationships are linear; some contain nonlinear patterns and geographic clusters.

Latitude and longitude provide location information. Even when their simple correlations are moderate, geographic variables can capture spatial effects such as proximity to desirable locations, the coast, major cities, and employment centers.

The relationship between median income and house value deserves deeper investigation because it is strong but not perfectly linear.


# Part 4 — Investment clients

For Clients A, B, and C, percentile ranks are used to create comparable scores. A weighted ranking is preferable to arbitrary hard thresholds because it preserves more observations and makes the priorities explicit.


## Client A — Budget-Oriented Investor

**Objective:** relatively low house value, higher local income, and lower occupancy.

**Weights:**
- Price: 45% — minimize
- Income: 40% — maximize
- Occupancy: 15% — minimize


In [ ]:
client_a = df.copy()

client_a["PriceScore"] = 1 - client_a["MedHouseVal"].rank(pct=True)
client_a["IncomeScore"] = client_a["MedInc"].rank(pct=True)
client_a["OccupancyScore"] = 1 - client_a["AveOccup"].rank(pct=True)

client_a["Score_A"] = (
    0.45 * client_a["PriceScore"] +
    0.40 * client_a["IncomeScore"] +
    0.15 * client_a["OccupancyScore"]
)

top_A = client_a.sort_values("Score_A", ascending=False).head(10)

display(top_A[[
    "Latitude", "Longitude", "MedHouseVal",
    "MedInc", "AveOccup", "Score_A"
]])


**Justification:** Price receives the largest weight because the client is budget-oriented. Income is almost as important because the client wants economically stronger areas. Occupancy is a secondary preference.


## Client B — Premium Market Investor

**Objective:** high median income, high house value, and larger homes.

**Weights:**
- Income: 40% — maximize
- House value: 40% — maximize
- Rooms: 20% — maximize


In [ ]:
client_b = df.copy()

client_b["IncomeScore"] = client_b["MedInc"].rank(pct=True)
client_b["PriceScore"] = client_b["MedHouseVal"].rank(pct=True)
client_b["RoomScore"] = client_b["AveRooms"].rank(pct=True)

client_b["Score_B"] = (
    0.40 * client_b["IncomeScore"] +
    0.40 * client_b["PriceScore"] +
    0.20 * client_b["RoomScore"]
)

top_B = client_b.sort_values("Score_B", ascending=False).head(10)

display(top_B[[
    "Latitude", "Longitude", "MedHouseVal",
    "MedInc", "AveRooms", "Score_B"
]])


**Justification:** Income and house value receive the highest weights because the client wants exposure to strong, expensive markets. Average rooms receives a lower weight because size is an additional preference.


## Client C — Space-Oriented Investor

**Objective:** more rooms, lower occupancy, and reasonable price.

**Weights:**
- Rooms: 45% — maximize
- Occupancy: 35% — minimize
- Price: 20% — minimize


In [ ]:
client_c = df.copy()

client_c["RoomScore"] = client_c["AveRooms"].rank(pct=True)
client_c["OccupancyScore"] = 1 - client_c["AveOccup"].rank(pct=True)
client_c["PriceScore"] = 1 - client_c["MedHouseVal"].rank(pct=True)

client_c["Score_C"] = (
    0.45 * client_c["RoomScore"] +
    0.35 * client_c["OccupancyScore"] +
    0.20 * client_c["PriceScore"]
)

top_C = client_c.sort_values("Score_C", ascending=False).head(10)

display(top_C[[
    "Latitude", "Longitude", "MedHouseVal",
    "AveRooms", "AveOccup", "Score_C"
]])


**Justification:** Rooms receive the highest weight because space is the primary objective. Lower occupancy is also important because it indicates less household crowding. Price receives a lower weight because the client wants a reasonable price rather than necessarily the cheapest area.


# Part 5 — Feature engineering

## Feature 1 — Rooms per occupant

**Formula:** `AveRooms / AveOccup`

**Meaning:** an approximate indicator of available room space relative to household occupancy.

**Limitation:** it is calculated from aggregated census variables and does not represent the actual usable area of an individual property.

## Feature 2 — Bedroom-to-room ratio

**Formula:** `AveBedrms / AveRooms`

**Meaning:** proportion of average rooms that are bedrooms.

**Limitation:** it does not capture bathrooms, floor area, property condition, or layout quality.


In [ ]:
df["RoomsPerOccupant"] = df["AveRooms"] / df["AveOccup"]
df["BedroomRatio"] = df["AveBedrms"] / df["AveRooms"]

display(df[[
    "AveRooms", "AveOccup", "RoomsPerOccupant",
    "AveBedrms", "BedroomRatio"
]].head())


# Part 6 — Predictive modeling

A 80/20 train-test split is used. The baseline predicts the mean training target. Four models are evaluated using MAE, RMSE, and R².


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Use the engineered features as predictors
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Baseline
baseline_pred = np.full(len(y_test), y_train.mean())

models = {
    "Baseline": None,
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
}

predictions = {"Baseline": baseline_pred}
results = []

results.append({
    "Model": "Baseline",
    "MAE": mean_absolute_error(y_test, baseline_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, baseline_pred)),
    "R2": r2_score(y_test, baseline_pred)
})

for name, model in list(models.items())[1:]:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)


### Answers

**1. Why is a baseline necessary?**  
It provides a reference point and shows whether the predictive models add useful information.

**2. Which model performs best?**  
Use the generated `results_df`. The preferred model has lower MAE/RMSE and higher R². In a typical California Housing analysis, Random Forest is expected to outperform the simple linear model.

**3. Does the best predictive model necessarily provide the best explanation?**  
No. A complex model may predict better but be harder to interpret.

**4. Which metric is most useful?**  
MAE is particularly useful for business communication because it expresses the average error directly in the target's units.


# Part 7 — Model interpretation

Random Forest feature importance is used as a simple interpretation method.


In [ ]:
rf_model = models["Random Forest"]

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)

plt.figure(figsize=(9, 6))
plt.barh(importance["Feature"], importance["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.show()


### Interpretation

Median income and geographic variables are expected to be among the most influential predictors. This is broadly consistent with the EDA.

Feature importance is not causal evidence. A feature can be useful for prediction without causing the target to change.

**Potentially misleading interpretation:** saying that a highly important variable "causes" higher house prices. The model identifies predictive relationships, not causal effects.


# Part 8 — Client D — Model-based value screening

A positive `PredictionGap` means the model predicts a higher value than the observed value. These observations can be screened for further investigation, but the gap is not proof of undervaluation.


In [ ]:
# Generate predictions for all observations using the best model
best_model_name = results_df.iloc[0]["Model"]

if best_model_name == "Baseline":
    all_predictions = np.full(len(df), y_train.mean())
else:
    best_model = models[best_model_name]
    all_predictions = best_model.predict(X)

df["PredictedValue"] = all_predictions
df["PredictionGap"] = df["PredictedValue"] - df["MedHouseVal"]

# Rank by positive prediction gap
client_d = df.sort_values("PredictionGap", ascending=False).copy()

display(client_d[[
    "Latitude", "Longitude", "MedHouseVal",
    "PredictedValue", "PredictionGap", "MedInc", "AveOccup"
]].head(20))


In [ ]:
# Additional business filter:
# income at or above median and occupancy at or below the 75th percentile

income_threshold = df["MedInc"].median()
occupancy_threshold = df["AveOccup"].quantile(0.75)

candidates_D = client_d[
    (client_d["PredictionGap"] > 0) &
    (client_d["MedInc"] >= income_threshold) &
    (client_d["AveOccup"] <= occupancy_threshold)
].copy()

top_D = candidates_D.head(10)

display(top_D[[
    "Latitude", "Longitude", "MedHouseVal",
    "PredictedValue", "PredictionGap",
    "MedInc", "AveOccup"
]])


### Answers

1. A large positive prediction gap is not proof of undervaluation because the model can be wrong.
2. The gap can result from missing variables, unusual local conditions, data quality issues, model limitations, or target capping.
3. Before investing, additional data should include rental prices, historical prices, taxes, mortgage rates, crime, schools, property size, vacancy, employment, and accessibility.


# Part 9 — Error analysis

In [ ]:
# Use the best non-baseline model
model_name = results_df[results_df["Model"] != "Baseline"].iloc[0]["Model"]
best_pred = predictions[model_name]

error_df = X_test.copy()
error_df["Observed"] = y_test.values
error_df["Predicted"] = best_pred
error_df["Error"] = error_df["Predicted"] - error_df["Observed"]
error_df["AbsoluteError"] = error_df["Error"].abs()

print("Best non-baseline model:", model_name)

print("\nLargest absolute errors:")
display(error_df.sort_values("AbsoluteError", ascending=False).head(20))

median_target = y_train.median()
error_df["PriceGroup"] = np.where(
    error_df["Observed"] < median_target,
    "Lower-value",
    "Higher-value"
)

print("\nErrors by price group:")
display(error_df.groupby("PriceGroup").agg(
    MAE=("AbsoluteError", "mean"),
    MeanError=("Error", "mean"),
    Count=("Error", "size")
))

plt.figure(figsize=(9, 7))
plt.scatter(
    error_df["Longitude"],
    error_df["Latitude"],
    c=error_df["AbsoluteError"],
    alpha=0.5
)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geographical Distribution of Prediction Errors")
plt.colorbar(label="Absolute Error")
plt.show()


### Interpretation

The table and geographic plot should be used to determine whether errors are concentrated in particular regions. Comparing lower-value and higher-value groups helps identify whether the model behaves differently across the price distribution.

The target cap can contribute to larger errors for high-value areas. In practice, this means the model should be used as a screening tool rather than as an automatic investment decision system.


# Part 10 — Final investment recommendations

The following cells automatically display the top five candidate areas for each client. Latitude and longitude are used because the assignment does not require city names.


In [ ]:
print("CLIENT A — TOP 5")
display(top_A[[
    "Latitude", "Longitude", "MedHouseVal",
    "MedInc", "AveOccup", "Score_A"
]].head(5))

print("CLIENT B — TOP 5")
display(top_B[[
    "Latitude", "Longitude", "MedHouseVal",
    "MedInc", "AveRooms", "Score_B"
]].head(5))

print("CLIENT C — TOP 5")
display(top_C[[
    "Latitude", "Longitude", "MedHouseVal",
    "AveRooms", "AveOccup", "Score_C"
]].head(5))

print("CLIENT D — TOP 5")
display(top_D[[
    "Latitude", "Longitude", "MedHouseVal",
    "PredictedValue", "PredictionGap",
    "MedInc", "AveOccup"
]].head(5))


### Recommendation framework

**Client A — Budget-Oriented:** choose the five highest-ranked areas using low value, high income, and low occupancy.  
**Risk:** low price may reflect unobserved negative characteristics.

**Client B — Premium:** choose the five highest-ranked areas using high income, high house value, and more rooms.  
**Risk:** a high current value does not guarantee future appreciation or investment return.

**Client C — Space-Oriented:** choose the five highest-ranked areas using more rooms, lower occupancy, and reasonable value.  
**Risk:** average rooms is an aggregated census measure, not individual property floor area.

**Client D — Value:** choose the five largest positive prediction gaps after the income/occupancy filter.  
**Risk:** a prediction gap can reflect model error or missing variables rather than genuine undervaluation.


# Part 11 — Critical discussion

## Question 1 — Current value vs appreciation vs investment return

Predicting current house value means estimating today's median value from observed characteristics. Predicting future appreciation means estimating how prices may change over time. Predicting investment return is broader and should incorporate appreciation, rental income, financing, taxes, maintenance, vacancy, and transaction costs.

Therefore, current value, future appreciation, and investment return are different objectives.

## Question 2 — Why is the dataset insufficient?

The dataset does not contain many variables needed for a real investment decision, including rental prices, historical prices, mortgage rates, property taxes, crime, school quality, property size, vacancy, maintenance costs, and accessibility.

## Question 3 — Why keep Linear Regression if Random Forest is better?

First, Linear Regression is easier to interpret because coefficients provide a simple description of the relationship between predictors and the target. Second, it is simple and provides a transparent benchmark against more complex models.

## Question 4 — Why does high predicted value not imply good investment?

A high predicted house value only indicates an expected level of current value based on the model. It does not establish future appreciation, rental yield, affordability, risk, financing costs, or expected return.

## Question 5 — How would additional data improve the project?

Rental prices would allow rental-yield analysis. Historical prices would allow appreciation and volatility analysis. Mortgage rates would improve financing and leveraged-return calculations. Property taxes would improve cash-flow estimates. Crime and school data would improve location-risk analysis. Distance from major cities could be used to model accessibility and demand.


# Optional Bonus A — Cross-validation

Five-fold cross-validation is used to compare Linear Regression and Random Forest.


In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
}

cv_results = []

for name, model in cv_models.items():
    scores = -cross_val_score(
        model,
        X,
        y,
        cv=kf,
        scoring="neg_root_mean_squared_error"
    )

    cv_results.append({
        "Model": name,
        "Mean RMSE": scores.mean(),
        "Std RMSE": scores.std()
    })

cv_results = pd.DataFrame(cv_results)
display(cv_results)


# Final conclusion

The analysis demonstrates how the California Housing dataset can be used for exploratory analysis, client segmentation, and predictive screening. Median income and geographic information are important predictors of median house value, while nonlinear models can capture relationships that a simple linear model cannot.

However, the dataset is not sufficient for making actual investment decisions. The analysis should therefore be considered a **first-stage screening framework**. Further investment analysis should incorporate rental income, historical price changes, financing costs, taxes, risk indicators, property-level characteristics, and other local market information.
